In [1]:
# # --- CLEAN INSTALL OF PYTORCH IN COLAB ---

# !pip uninstall -y torch torchvision torchaudio
# !pip cache purge

# # Install stable CUDA 12.1 wheels (Colab default)
# !pip install torch==2.3.1 torchvision==0.18.1 torchaudio==2.3.1 --index-url https://download.pytorch.org/whl/cu121

# # Install ONNX and ORT
# !pip install onnx==1.15.0 onnxruntime==1.19.2

# print("Done. Now run the next cell to restart the runtime.")

In [2]:
# import os
# import signal
# import time

# print("Restarting runtime...")
# time.sleep(1)

# # This kills the current process and forces Colab to restart
# os.kill(os.getpid(), signal.SIGKILL)

In [3]:
import os
import sys
import subprocess
import zipfile
import json
from tqdm.auto import tqdm
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
# set globals
USER = 'josemarquezjaramillo'
REPO = 'crypto-rl-portfolio'
DATASET = 'dataset_v1'

In [5]:
# pull repo - colab starts uninitialized
os.system(f'git clone https://github.com/{USER}/{REPO}')
os.chdir(REPO)

# install requirements
req_result = subprocess.run('pip install -r requirements-data.txt', shell=True, capture_output=True, text=True)

print("=== STDOUT ===")
print(req_result.stdout.strip())   # remove extra blank lines
print("\n=== STDERR ===")
print(req_result.stderr.strip())

print(f"\nExit code: {req_result.returncode}")

=== STDOUT ===
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.3/252.3 kB 30.2 MB/s eta 0:00:00

=== STDERR ===


Exit code: 0


In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.7/404.7 kB 32.3 MB/s eta 0:00:00


In [7]:
from agents.policy_grad.policygrad import PolicyGradAgent, PolicyGradConfig

In [8]:
import zipfile

zip_file_path = 'data/dataset_v1.zip'
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall()

In [9]:
import sys
from pathlib import Path
import argparse
import numpy as np
import torch
import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv
import signal
from contextlib import contextmanager

from torch import nn

# Load environment variables
load_dotenv()

# Add project root to path
# project_root = Path(__file__).parent.parent.parent
# sys.path.insert(0, str(project_root))

from data.data_loader import DatabaseConfig
from data.dataset_loader import load_exported_dataset
from data.dataset_backend import DatasetBackend
from environment.environment import PortfolioEnv, EnvConfig

from agents.policy_grad.policygrad import PolicyGradAgent, PolicyGradConfig
from agents.dqn.hyperparameter_search import create_environments

In [10]:
# Hyperparameter search space
SEARCH_CONFIG = {
    "learning_rate": (1e-6, 1e-4),
    "recurrent_layer": ["GRU", "LSTM", "RNN"],   # <-- strings now
    "epsilon_decay_episodes": [300, 500, 1000],
    "epsilon_end": [0.01, 0.05, 0.1],
    "hidden_dim": [128, 256, 512],
    "temperature": [1, 2, 5],
}

RNN_LAYER_MAP = {
    "GRU": nn.GRU,
    "LSTM": nn.LSTM,
    "RNN": nn.RNN,
}

# Training configuration
TRAINING_CONFIG = {
    # 'n_training_episodes': 50,  # Episodes per trial
    'n_val_episodes': 5,  # Validation episodes (one per regime window)
    'patience': 10,
    'validation_freq': 50,
    'window_length': 100,
    'max_episodes': 2000,
    'min_episodes': 500,
}

# Environment configuration
ENV_CONFIG = {
    'cost_rate': 0.001,
    'turnover_cap': 0.30,
    'max_weight_per_asset': 0.35,
    'strict_projection': False,
    'constraint_penalty': -10.0,
    'terminate_on_violation': False,
}

In [11]:
import json, pickle, numpy as np
from pathlib import Path
import torch
import torch.nn as nn

def train_with_early_stopping_pg(
    agent,
    ds,
    train_backend,
    val_env,
    window_length: int = 100,
    run_dir: Path = None,
    trial=None,
    resume: bool = False,
    backup_every: int = 50,        # change to None to disable periodic backups
):
    checkpoint_dir = run_dir or Path("./checkpoints/run")
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    best_dir = checkpoint_dir / "best"
    best_dir.mkdir(exist_ok=True)

    log_file = checkpoint_dir / "training_log.csv"
    if not log_file.exists():
        with open(log_file, "w") as f:
            f.write("episode,train_return,epsilon,val_return,val_std,is_best\n")

    # Sliding window setup
    all_train_dates = train_backend.dates()
    total_days = len(all_train_dates)
    max_start_day = total_days - window_length

    best_val_return = -np.inf
    episodes_since_improvement = 0
    validation_history = []

    print("\n=== PG TRAINING (Sliding Windows) ===\n")

    for episode in range(TRAINING_CONFIG["max_episodes"]):

        # -----------------------------
        # Sample training window
        # -----------------------------
        start_idx = np.random.randint(0, max_start_day + 1)
        end_idx = start_idx + window_length - 1

        window_backend = DatasetBackend(
            ds,
            split_tag_filter="train_core",
            start_date=str(all_train_dates[start_idx]),
            end_date=str(all_train_dates[end_idx])
        )

        window_env = PortfolioEnv(
            EnvConfig(split="train", random_seed=None, **ENV_CONFIG),
            window_backend
        )

        agent.env = window_env
        episode_metrics = agent.train_episode()

        train_return = episode_metrics.total_reward
        epsilon = agent.epsilon

        # print(f"[Episode {episode+1}] Return={train_return:.4f}, eps={epsilon:.3f}")

        # -------------------------------------
        # VALIDATION CHECK
        # -------------------------------------
        val_return = None
        val_std = None
        is_best = False

        is_validation_episode = ((episode + 1) % TRAINING_CONFIG["validation_freq"] == 0)

        if is_validation_episode:

            val_metrics = agent.evaluate_on_env(
                val_env,
                n_episodes=TRAINING_CONFIG["n_val_episodes"],
                deterministic=True
            )

            val_return = val_metrics["mean_return"]
            val_std = val_metrics["std_return"]

            print(f"   → Validation: {val_return:.6f} ± {val_std:.6f}")

            # -------------------------------------
            # IMPROVEMENT CHECK (save only if better)
            # -------------------------------------
            if (episode + 1) >= TRAINING_CONFIG["min_episodes"]:
                if val_return > best_val_return:
                    best_val_return = val_return
                    episodes_since_improvement = 0
                    is_best = True

                    print("   ✓ NEW BEST — Saving checkpoint")
                    agent.save(best_dir)

                else:
                    episodes_since_improvement += 1
                    print(f"   No improvement ({episodes_since_improvement}/{TRAINING_CONFIG['patience']})")

            # Optuna reporting / pruning
            if trial is not None:
                trial.report(val_return, episode)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            # Early stopping
            if ((episode + 1) >= TRAINING_CONFIG["min_episodes"] and
                episodes_since_improvement >= TRAINING_CONFIG["patience"]):
                print("\n--- EARLY STOPPING ---\n")
                break

        # -------------------------------------
        # OPTIONAL: periodic backup (non-best)
        # -------------------------------------
        if backup_every is not None and (episode + 1) % backup_every == 0:
            backup_dir = checkpoint_dir / f"backup_ep{episode+1}"
            backup_dir.mkdir(exist_ok=True)
            print(f"   → Saving periodic backup at ep {episode+1}")
            agent.save(backup_dir)

        # -------------------------------------
        # Log CSV
        # -------------------------------------
        with open(log_file, "a") as f:
            f.write(f"{episode+1},{train_return:.6f},{epsilon:.4f},")
            if val_return is not None:
                f.write(f"{val_return:.6f},{val_std:.6f},{int(is_best)}\n")
            else:
                f.write(",,\n")

    summary = {
        "best_val_return": float(best_val_return),
        "episodes": episode + 1,
        "validation_history": validation_history,
        "final_epsilon": float(agent.epsilon),
        "best_model_path": str(best_dir),
        "log_file": str(log_file),
    }

    with open(checkpoint_dir / "training_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    with open(checkpoint_dir / "training_summary.pkl", "wb") as f:
        pickle.dump(summary, f)

    return summary


In [12]:
def optuna_objective(trial, ds, train_backend, val_env):

    lr = trial.suggest_float(
        "learning_rate",
        SEARCH_CONFIG["learning_rate"][0],
        SEARCH_CONFIG["learning_rate"][1],
        log=True
    )

    hidden_dim = trial.suggest_categorical("hidden_dim", SEARCH_CONFIG["hidden_dim"])
    temperature = trial.suggest_categorical("temperature", SEARCH_CONFIG["temperature"])
    epsilon_decay = trial.suggest_categorical("epsilon_decay_episodes",
                                              SEARCH_CONFIG["epsilon_decay_episodes"])
    epsilon_end = trial.suggest_categorical("epsilon_end", SEARCH_CONFIG["epsilon_end"])

    # recurrent layer: choose a STRING
    rnn_layer_name = trial.suggest_categorical(
        "recurrent_layer",
        SEARCH_CONFIG["recurrent_layer"]
    )
    rnn_layer = RNN_LAYER_MAP[rnn_layer_name]   # convert to actual class

    cfg = PolicyGradConfig(
        name="reinforce",
    )

    cfg.hidden_dim=hidden_dim
    cfg.learning_rate=lr
    cfg.recurrent_layer=rnn_layer
    cfg.epsilon_decay_episodes=epsilon_decay
    cfg.epsilon_end=epsilon_end
    cfg.temperature=temperature
    cfg.epsilon_start=1.0
    cfg.device="cuda"
    cfg.dataset_path="dataset_v1"

    temp_env = PortfolioEnv(
        EnvConfig(split="train", random_seed=42, **ENV_CONFIG),
        train_backend
    )

    agent = PolicyGradAgent(cfg, temp_env)

    trial_dir = Path(f"./optuna_runs/trial_{trial.number}")
    trial_dir.mkdir(parents=True, exist_ok=True)

    summary = train_with_early_stopping_pg(
        agent,
        ds,
        train_backend,
        val_env,
        window_length=TRAINING_CONFIG["window_length"],
        run_dir=trial_dir,
        trial=trial
    )

    return summary["best_val_return"]


In [13]:
from tqdm.auto import tqdm

def run_study(ds, train_backend, val_env, n_trials=30):

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
        study_name="reinforce_hyperparam_search",
        storage="sqlite:///reinforce_optuna.db",
        load_if_exists=True
    )

    print("\n=== Running Optuna Hyperparameter Search ===\n")

    # tqdm progress bar
    pbar = tqdm(total=n_trials, desc="Optuna Trials", position=0)

    def wrapped_objective(trial):
        value = optuna_objective(trial, ds, train_backend, val_env)

        # update tqdm display with best-so-far
        if study.best_trial is not None:
            pbar.set_postfix({
                "best": f"{study.best_value:.4f}"
            })

        pbar.update(1)
        return value

    study.optimize(
        wrapped_objective,
        n_trials=n_trials,
        gc_after_trial=True,
    )

    pbar.close()

    print("\n=== OPTUNA COMPLETE ===")
    print("Best Params:", study.best_params)
    print("Best Value:", study.best_value)

    # save final result
    with open("best_params.json", "w") as f:
        json.dump(study.best_params, f, indent=2)

    return study


In [14]:
ds, train_backend, val_env = create_environments(1)

In [ ]:
study = run_study(
    ds=ds,
    train_backend=train_backend,
    val_env=val_env,
    n_trials=20
)

[I 2025-12-05 22:54:37,851] A new study created in RDB with name: reinforce_hyperparam_search



=== Running Optuna Hyperparameter Search ===



Optuna Trials:   0%|          | 0/20 [00:00<?, ?it/s]


=== PG TRAINING (Sliding Windows) ===

   → Validation: -0.947751 ± 0.000000
   → Saving periodic backup at ep 50
[PolicyGradAgent] Saved checkpoint to: optuna_runs/trial_0/backup_ep50
   → Validation: -0.947747 ± 0.000000
   → Saving periodic backup at ep 100
[PolicyGradAgent] Saved checkpoint to: optuna_runs/trial_0/backup_ep100
   → Validation: -0.947672 ± 0.000000
   → Saving periodic backup at ep 150
[PolicyGradAgent] Saved checkpoint to: optuna_runs/trial_0/backup_ep150
   → Validation: -0.947542 ± 0.000000
   → Saving periodic backup at ep 200
[PolicyGradAgent] Saved checkpoint to: optuna_runs/trial_0/backup_ep200
   → Validation: -0.947804 ± 0.000000
   → Saving periodic backup at ep 250
[PolicyGradAgent] Saved checkpoint to: optuna_runs/trial_0/backup_ep250
